# SARM Reward Model 預測視覺化（純測試）

本 notebook 用來載入**已訓練好**的 SARM Reward Model（從 HF Hub）並對標注資料集做預測視覺化。

- 訓練好的模型：`Lebruhbruh/sarm-charging-bimanual`
- 資料集：`Lebruhbruh/SARM-opendata-annotated-fixed`
- 對指定 episodes (21, 26, 47, 50, 229, 244) 計算 dense reward 並輸出 PNG

順序：GPU 確認 → 使用者設定 → 進度工具 → 安裝 LeRobot → Transformers 5.x patch → HF 登入 → Cell 14（產生視覺化）→ Cell 14b（顯示結果）

In [ ]:
# Cell 0: GPU 確認
import subprocess, torch
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
assert torch.cuda.is_available(), '錯誤：沒有 GPU！請檢查 Runtime 設定'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')
if vram_gb < 16:
    print('WARNING: VRAM < 16GB，推論可能 OOM，建議至少 T4 / L4')
else:
    print('✓ GPU 規格符合推論需求')

In [ ]:
# Cell 1: 使用者設定（必填）
# ============================================================
HF_USERNAME = 'Lebruhbruh'
HF_TOKEN    = ''   # <-- 貼上你的 HF Token（read 權限即可，因為這個 notebook 不會 push）

# 已標注好的資料集（從 Hub 讀）
ANNOTATED_DATASET = 'Lebruhbruh/SARM-opendata-annotated-fixed'

# 訓練好的 SARM 模型 repo（從 Hub 載）
MODEL_REPO_ID = f'{HF_USERNAME}/sarm-charging-bimanual'

# 主要訓練/視覺化用相機
VIDEO_KEY = 'observation.images.top'

# 四個子任務（subtask_index 0–3）
SUBTASK_NAMES = [
    'Pick up the phone',
    'Flip the phone sideways',
    'Pick up the charging cable and plug it into the phone',
    'Turn on the power of the extension cord',
]

# 視覺化參數
# 指定要 eval 的 episode index（PNG 與 MP4 共用）
EVAL_EPISODES      = [21, 26, 47, 50, 229, 244]
HEAD_MODE          = 'dense'
VIZ_OUTPUT_DIR     = '/content/sarm_predictions_viz'

print(f'標注資料集: {ANNOTATED_DATASET}')
print(f'SARM 模型:  {MODEL_REPO_ID}')
print(f'相機:       {VIDEO_KEY}')
print(f'視覺化:     episodes={EVAL_EPISODES}, head-mode={HEAD_MODE}')
print(f'輸出目錄:   {VIZ_OUTPUT_DIR}')
print('子任務:')
for i, t in enumerate(SUBTASK_NAMES):
    print(f'  [{i}] {t}')

In [ ]:
# Cell 1b: 進度回饋工具（後續長時 cell 共用）
# ============================================================
import threading, time, contextlib
from datetime import timedelta

def _fmt_elapsed(seconds: float) -> str:
    return str(timedelta(seconds=int(seconds)))

@contextlib.contextmanager
def progress_stage(title: str, steps=None, heartbeat_seconds: int = 30):
    print(f'\n━━━ {title} ━━━')
    if steps:
        for i, step in enumerate(steps, 1):
            print(f'  {i}) {step}')
        print()

    stop_evt = threading.Event()
    start = time.monotonic()

    def _heartbeat():
        while not stop_evt.wait(heartbeat_seconds):
            print(f'  ⏱  已執行 {_fmt_elapsed(time.monotonic() - start)}', flush=True)

    th = threading.Thread(target=_heartbeat, daemon=True)
    th.start()
    error = None
    try:
        yield
    except BaseException as e:
        error = e
        raise
    finally:
        stop_evt.set()
        th.join(timeout=1)
        total = _fmt_elapsed(time.monotonic() - start)
        if error is None:
            print(f'\n✓ {title} 完成，總耗時 {total}')
        else:
            print(f'\n✗ {title} 失敗（{type(error).__name__}），已執行 {total}')

print('✓ progress_stage 已就緒')

In [ ]:
# Cell 2: 安裝 LeRobot + SARM
import subprocess, sys

with progress_stage(
    '安裝 LeRobot + SARM（預計 2-5 分鐘）',
    steps=['pip install lerobot[sarm] from GitHub', '確認 lerobot / transformers 版本'],
    heartbeat_seconds=30,
):
    print('1/2 安裝 LeRobot + SARM...')
    r = subprocess.run(
        'pip install -q "git+https://github.com/huggingface/lerobot.git#egg=lerobot[sarm]"',
        shell=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f'pip install 失敗，返回碼 {r.returncode}')
    print('  ✓ LeRobot 安裝完成')

    print('2/2 確認安裝...')
    check_cmd = 'import lerobot, transformers; print(lerobot.__version__); print(transformers.__version__)'
    r = subprocess.run([sys.executable, '-c', check_cmd], capture_output=True, text=True)
    if r.returncode == 0:
        versions = r.stdout.strip().split()
        print(f'  LeRobot: {versions[0]}')
        print(f'  Transformers: {versions[1] if len(versions)>1 else "?"}')
    else:
        raise RuntimeError(f'版本確認失敗: {r.stderr}')

In [ ]:
# Cell 3: 修補 Transformers 5.x Bug（CRITICAL）
import os, glob, torch, subprocess
import lerobot

lerobot_root = os.path.dirname(lerobot.__file__)
print(f"LeRobot 安裝路徑: {lerobot_root}")

hits = glob.glob(os.path.join(lerobot_root, "**", "processor_sarm.py"), recursive=True)
if not hits:
    print("lerobot_root 下找不到，往上一層搜尋...")
    hits = glob.glob(os.path.join(os.path.dirname(lerobot_root), "**", "processor_sarm.py"), recursive=True)

if not hits:
    raise FileNotFoundError(
        "找不到 processor_sarm.py，請確認 lerobot[sarm] 安裝成功。"
        "可在新 cell 執行: !find /usr/local/lib -name processor_sarm.py"
    )

processor_path = hits[0]
print(f"修補目標: {processor_path}")

with open(processor_path) as f:
    content = f.read()

patched = False

indent = "\n        "
old_img = "embeddings = self.clip_model.get_image_features(**inputs).detach().cpu()"
new_img = indent.join([
    "output = self.clip_model.get_image_features(**inputs)",
    "if not isinstance(output, torch.Tensor):",
    "    output = output.pooler_output",
    "embeddings = output.detach().cpu()",
])
if old_img in content:
    content = content.replace(old_img, new_img)
    print("✓ 已修補 image features")
    patched = True
else:
    print("ℹ image features 不需修補")

old_txt = "embeddings = self.clip_model.get_text_features(**inputs).detach().cpu()"
new_txt = indent.join([
    "output = self.clip_model.get_text_features(**inputs)",
    "if not isinstance(output, torch.Tensor):",
    "    output = output.pooler_output",
    "embeddings = output.detach().cpu()",
])
if old_txt in content:
    content = content.replace(old_txt, new_txt)
    print("✓ 已修補 text features")
    patched = True
else:
    print("ℹ text features 不需修補")

with open(processor_path, "w") as f:
    f.write(content)

if patched:
    r = subprocess.run(["grep", "-n", "pooler_output", processor_path],
                       capture_output=True, text=True)
    print("\n驗證修補（應看到 pooler_output）:")
    print(r.stdout)
else:
    print("\n✓ 程式碼已是最新版，無需修補")

In [ ]:
# Cell 4: HuggingFace 登入
from huggingface_hub import login
assert HF_TOKEN, '請先在 Cell 1 填入 HF_TOKEN'
login(token=HF_TOKEN)
print('✓ HuggingFace 登入成功')

In [ ]:
# Cell 4b: 確認模型 repo 存在
from huggingface_hub import HfApi
api = HfApi()
try:
    info = api.model_info(MODEL_REPO_ID)
    print(f'✓ 模型 repo 存在: {MODEL_REPO_ID}')
    print(f'  last_modified: {info.last_modified}')
    siblings = [s.rfilename for s in info.siblings]
    print(f'  files ({len(siblings)}):')
    for f in siblings:
        print(f'    - {f}')
except Exception as e:
    raise RuntimeError(f'找不到模型 repo {MODEL_REPO_ID}: {e}')

In [ ]:
# Cell 4c: 處理「dataset 沒有 v3.0 tag」的情況
# ============================================================
# HGLLL/SARM-opendata-annotated-1 這類由他人上傳的 dataset 常常沒有打
# `v3.0` codebase_version tag，會導致 LeRobotDataset 初始化時：
#   FileNotFoundError → get_safe_version() → RevisionNotFoundError
#   （甚至包裝成 TypeError：HfHubHTTPError missing 'response'）
#
# 兩步處理：
#   1) 先嘗試在 Hub 上補打 `v3.0` tag（需要該 repo 的 write 權限，多半會 403）
#   2) 失敗就 monkey-patch `get_safe_version`，無 tag 時 fallback 到 'main'
# ============================================================
from huggingface_hub import HfApi

api = HfApi()
tag_ok = False
try:
    api.create_tag(ANNOTATED_DATASET, tag='v3.0', repo_type='dataset', exist_ok=True)
    print(f'✓ 已在 {ANNOTATED_DATASET} 上補打 v3.0 tag')
    tag_ok = True
except Exception as e:
    print(f'ℹ 無法在 Hub 上補 tag（{type(e).__name__}: {str(e)[:120]}）')
    print('  → 改用 monkey-patch fallback 到 "main"')

if not tag_ok:
    import lerobot.datasets.utils as _ds_utils
    import lerobot.datasets.dataset_metadata as _dsmeta

    _orig_get_safe_version = _ds_utils.get_safe_version

    def _patched_get_safe_version(repo_id, version):
        try:
            return _orig_get_safe_version(repo_id, version)
        except Exception as e:
            print(f'  ⚠ get_safe_version({repo_id}) 失敗（{type(e).__name__}），fallback → "main"')
            return 'main'

    _ds_utils.get_safe_version = _patched_get_safe_version
    if hasattr(_dsmeta, 'get_safe_version'):
        _dsmeta.get_safe_version = _patched_get_safe_version
    print('✓ get_safe_version monkey-patch 已就緒')


In [ ]:
# Cell 4d: 修剪 info.json + 放寬 timestamp 容忍度
# ============================================================
# 兩個問題一起解：
#
# (A) HGLLL/SARM-opendata-annotated-1 的 info.json 宣告了 3 顆相機
#     (right / left / top)，但 Hub 上 right 沒有實檔，而且模型其實
#     只用 top（reward_model.config.image_key='observation.images.top'）。
#     LeRobotDataset 預設會載入所有宣告的 video feature → 既浪費 IO
#     又會在 right 直接 FileNotFoundError。
#     → 把 info.json 改成只留 model 真正用到的相機。
#
# (B) torchcodec 解 mp4 時 query / loaded timestamp 會有 FP 精度誤差，
#     LeRobotDataset 預設 tolerance_s=1e-4，邊界差 0.0001 也會炸：
#         queried 1848.2666 vs loaded 1848.2667 → FrameTimestampError
#     → monkey-patch LeRobotDataset.__init__ 把預設 tolerance 拉到 1e-2，
#       因為這個 dataset 是 30fps（frame interval 33ms），1e-2 仍遠小於
#       單幀間距，不會抓錯 frame。
# ============================================================
import json
import inspect
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
from lerobot.utils.constants import HF_LEROBOT_HUB_CACHE
import lerobot.datasets.lerobot_dataset as _lr_ds

MODEL_IMAGE_KEY = 'observation.images.top'  # 從 Lebruhbruh/sarm-charging-bimanual config.json
NEEDED_VIDEO_FEATURES = {MODEL_IMAGE_KEY}

api = HfApi()

# --- (A) 修 info.json ---
meta_dir = Path(snapshot_download(
    repo_id=ANNOTATED_DATASET,
    repo_type='dataset',
    revision='main',
    allow_patterns=['meta/*'],
    cache_dir=HF_LEROBOT_HUB_CACHE,
))
info_path = meta_dir / 'meta' / 'info.json'
print(f'info.json: {info_path}')

# 列 Hub 上實際存在的相機目錄
tree = api.list_repo_tree(
    repo_id=ANNOTATED_DATASET, repo_type='dataset',
    path_in_repo='videos', revision='main', recursive=False,
)
present_cams = {Path(e.path).name for e in tree if Path(e.path).name.startswith('observation.images')}
print(f'Hub 實際有影片的相機: {sorted(present_cams)}')

info = json.loads(info_path.read_text())
declared_cams = {k for k, v in info['features'].items()
                 if k.startswith('observation.images') and v.get('dtype') == 'video'}
# 目標 = (Hub 上有實檔) ∩ (模型需要的)
keep_cams = declared_cams & present_cams & NEEDED_VIDEO_FEATURES
drop_cams = declared_cams - keep_cams
print(f'info.json 宣告的相機: {sorted(declared_cams)}')
print(f'保留:           {sorted(keep_cams)}')
print(f'移除:           {sorted(drop_cams)}')

assert MODEL_IMAGE_KEY in keep_cams, \
    f'⚠ 模型需要的 {MODEL_IMAGE_KEY} 沒被保留（dataset 沒有 / Hub 上沒實檔）'

if drop_cams:
    for cam in drop_cams:
        info['features'].pop(cam, None)
    info_path.write_text(json.dumps(info, indent=4))
    print(f'✓ 已從 info.json 移除 {len(drop_cams)} 個相機 feature')
else:
    print('✓ info.json 不需修改')

# --- (B) 放寬 LeRobotDataset 預設 tolerance_s ---
_OrigInit = _lr_ds.LeRobotDataset.__init__
_orig_sig = inspect.signature(_OrigInit)

def _patched_init(self, *args, **kwargs):
    kwargs.setdefault('tolerance_s', 1e-2)  # 30fps → 33ms 一幀，1e-2 仍安全
    return _OrigInit(self, *args, **kwargs)

_patched_init.__signature__ = _orig_sig
_lr_ds.LeRobotDataset.__init__ = _patched_init
print('✓ LeRobotDataset 預設 tolerance_s 改為 1e-2')


In [ ]:
# Cell 13z: 診斷「為何 predict 全部 0%」
# ============================================================
# 對 EVAL_EPISODES[0] 取「episode 中段」單一 frame 跑一次完整 inference，
# 把以下都印出來，幫忙定位是哪一段歪掉：
#   1) dataset 的 task 文字（CLIP 輸入）
#   2) 模型訓練時的 subtask_names（model config）
#   3) 模型訓練時的 stats vs 當前 dataset 的 stats（state normalization 是否錯位）
#   4) preprocess 出來的 video / text / state features shape + 統計
#   5) 不同 head_mode 的 reward 原始輸出
# ============================================================
import torch, numpy as np, json
from lerobot.rewards.sarm.compute_rabc_weights import load_sarm_resources

dataset, reward_model, preprocess = load_sarm_resources(
    ANNOTATED_DATASET, MODEL_REPO_ID, device='cuda',
)

cfg = reward_model.config
image_key = cfg.image_key
state_key = cfg.state_key
target_idx = cfg.n_obs_steps // 2

print('='*70)
print('1) Model config 重點')
print('='*70)
print(f'  image_key:        {cfg.image_key}')
print(f'  state_key:        {cfg.state_key}')
print(f'  n_obs_steps:      {cfg.n_obs_steps}  (target_idx={target_idx})')
print(f'  uses_dual_heads:  {cfg.uses_dual_heads}')
print(f'  num_sparse_stages:{cfg.num_sparse_stages}')
print(f'  num_dense_stages: {cfg.num_dense_stages}')
print(f'  sparse_subtask_names: {cfg.sparse_subtask_names}')
print(f'  dense_subtask_names:  {cfg.dense_subtask_names}')

# 在 preprocess.eval() 模式
if hasattr(preprocess, 'eval'):
    preprocess.eval()
for step in getattr(preprocess, 'steps', []):
    if hasattr(step, 'eval'):
        step.eval()

# 找 EVAL_EPISODES[0] 中段那個 frame
ep_idx = EVAL_EPISODES[0]
ep = dataset.meta.episodes[ep_idx]
ep_start = int(ep['dataset_from_index'])
ep_end   = int(ep['dataset_to_index'])
mid = (ep_start + ep_end) // 2
sample = dataset[mid]
task = sample.get('task', 'perform the task')

print()
print('='*70)
print(f'2) Dataset 端：episode {ep_idx} 中段 frame {mid}')
print('='*70)
print(f'  task 文字: {repr(task)}')
print(f'  episode 長度: {ep_end - ep_start} frames')
print(f'  sample keys: {list(sample.keys())}')

# state stats
if state_key in dataset.meta.stats:
    sk_stats = dataset.meta.stats[state_key]
    print(f'  當前 dataset {state_key} stats:')
    for k, v in sk_stats.items():
        if hasattr(v, 'tolist'):
            v = v.tolist()
        if isinstance(v, list) and len(v) > 6:
            v = f'[{v[0]:.3f}, {v[1]:.3f}, ..., {v[-1]:.3f}] (len={len(v)})'
        print(f'    {k}: {v}')

# 模型訓練時是否帶有 stats？
trained_stats = getattr(cfg, 'dataset_stats', None) or getattr(cfg, 'normalization_stats', None)
if trained_stats:
    print(f'  模型 config 自帶 stats keys: {list(trained_stats.keys()) if isinstance(trained_stats, dict) else type(trained_stats)}')
else:
    print(f'  模型 config 沒帶訓練時 stats → preprocess 會用「當前 dataset」的 stats normalize')
    print(f'    ⚠ 如果當前 dataset 的 state 範圍跟訓練時不同 → state 進模型會 out-of-distribution')

print()
print('='*70)
print('3) Preprocess 出來的 features')
print('='*70)
batch = {image_key: sample[image_key], 'task': task,
         'index': mid, 'episode_index': ep_idx}
if state_key in sample:
    batch[state_key] = sample[state_key]

with torch.no_grad():
    processed = preprocess(batch)

for k in ['video_features', 'text_features', 'state_features']:
    v = processed.get(k)
    if v is None:
        print(f'  {k}: None')
        continue
    if isinstance(v, torch.Tensor):
        arr = v.detach().cpu().numpy()
        print(f'  {k}: shape={tuple(arr.shape)}, dtype={arr.dtype}, '
              f'min={arr.min():.4f}, max={arr.max():.4f}, '
              f'mean={arr.mean():.4f}, std={arr.std():.4f}, '
              f'nonzero%={(arr != 0).mean()*100:.1f}')

print()
print('='*70)
print('4) Reward 原始輸出（sparse + dense）')
print('='*70)
for hm in (['sparse', 'dense'] if cfg.uses_dual_heads else ['sparse']):
    with torch.no_grad():
        vf = processed['video_features'].to(reward_model.device)
        tf = processed['text_features'].to(reward_model.device)
        sf = processed.get('state_features')
        if sf is not None:
            sf = sf.to(reward_model.device)
        lengths = processed.get('lengths')
        reward, stages = reward_model.calculate_rewards(
            text_embeddings=tf, video_embeddings=vf,
            state_features=sf, lengths=lengths,
            return_all_frames=True, return_stages=True,
            head_mode=hm,
        )
    if isinstance(reward, torch.Tensor):
        reward = reward.cpu().numpy()
        stages = stages.cpu().numpy()
    if reward.ndim == 2:
        r = reward[0, target_idx]; s = stages[0, target_idx, :]
    else:
        r = reward[target_idx]; s = stages[target_idx, :]
    print(f'  [{hm}] reward (this frame, target_idx={target_idx}) = {float(r):.4f}')
    print(f'        stage_probs = {np.round(s, 4).tolist()}')
    print(f'        reward 整段 shape={reward.shape}, min={reward.min():.4f}, '
          f'max={reward.max():.4f}, mean={reward.mean():.4f}')

print()
print('='*70)
print('結論判讀提示：')
print('  - 若 state_features 統計值極端（abs > 5 或全是 0）→ state normalization 問題')
print('  - 若 reward 任何 frame 都接近 0 但 stage_probs 有變化 → progress head 塌了')
print('  - 若 stage_probs 全部均等 (~0.25) → text/video embedding 沒有分辨力')
print('='*70)


In [ ]:
# Cell 14: 對指定 episodes 產生 PNG 視覺化（in-process）
# 注意：lerobot.rewards.sarm.compute_rabc_weights 的 CLI 只支援
#       --num-visualizations N（取 range(N)），無法指定任意 episode index。
#       因此直接呼叫底層 visualize_sarm_predictions() 並傳入 EVAL_EPISODES。
import os
from pathlib import Path

from lerobot.rewards.sarm.compute_rabc_weights import (
    load_sarm_resources,
    visualize_sarm_predictions,
)

os.makedirs(VIZ_OUTPUT_DIR, exist_ok=True)

with progress_stage(
    f'載入 SARM 資源（dataset={ANNOTATED_DATASET}, model={MODEL_REPO_ID}）',
    steps=['SARMRewardModel.from_pretrained',
           'LeRobotDataset（帶 delta_timestamps）',
           'make_sarm_pre_post_processors'],
    heartbeat_seconds=30,
):
    dataset, reward_model, preprocess = load_sarm_resources(
        ANNOTATED_DATASET, MODEL_REPO_ID, device='cuda',
    )

# 過濾掉超出資料集範圍的 episode index
valid_episodes = [e for e in EVAL_EPISODES if 0 <= e < dataset.num_episodes]
skipped = [e for e in EVAL_EPISODES if e not in valid_episodes]
if skipped:
    print(f'⚠ 跳過超出範圍的 episode（dataset 共 {dataset.num_episodes} 個）: {skipped}')
if not valid_episodes:
    raise RuntimeError(f'EVAL_EPISODES 中沒有任何有效 episode（dataset 只有 {dataset.num_episodes} 個）')

print(f'即將視覺化 episodes: {valid_episodes}')

with progress_stage(
    f'產生 SARM 預測 PNG（{len(valid_episodes)} episodes, head-mode={HEAD_MODE}）',
    steps=[
        f'對每個 episode 算 {HEAD_MODE} reward',
        f'輸出 PNG 到 {VIZ_OUTPUT_DIR}',
    ],
    heartbeat_seconds=30,
):
    visualize_sarm_predictions(
        dataset=dataset,
        reward_model=reward_model,
        preprocess=preprocess,
        episode_indices=valid_episodes,
        head_mode=HEAD_MODE,
        output_dir=Path(VIZ_OUTPUT_DIR),
        stride=1,
    )

print(f'\n✓ PNG 視覺化完成，存放於 {VIZ_OUTPUT_DIR}')


In [ ]:
# Cell 14b: 顯示預測結果
import glob, os
from IPython.display import Image as IPyImage, display

print('SARM Reward Model 預測結果:')
pred_files = sorted(glob.glob(f'{VIZ_OUTPUT_DIR}/*.png'))
if not pred_files:
    print('找不到預測圖片，請確認 Cell 14 成功執行')
else:
    for path in pred_files:
        print(f'\n{os.path.basename(path)}:')
        display(IPyImage(path))
    print('\n預期結果：每個 episode 的進度曲線應單調遞增（0 → 1）')
    print('         子任務轉換處應有明顯的斜率變化')

In [ ]:
# Cell 14c: 生成「影片 + 同步進度條」MP4（in-process 推論）
# ============================================================
# 版面：左半 = 原始 video frame
#       右半 = 上：水平 progress bar（當前 frame 的 predicted progress）
#              中：整個 episode 的 predicted progress 曲線 + 黑色 marker
#              下：當前主導 subtask 文字
# 速度：每 VIDEO_STRIDE 個 frame 推論 1 次，其他內插（VIDEO_STRIDE=1 最準但最慢）
# 預設使用 Cell 1 的 EVAL_EPISODES（與 PNG 視覺化相同的 episode 集合）
# ============================================================
import os, sys, importlib
import numpy as np
import torch
from tqdm.auto import tqdm
from pathlib import Path

VIZ_VIDEO_DIR     = '/content/sarm_predictions_video'
VIDEO_STRIDE       = 5      # 推論加速；改 1 = 每 frame 都推論
VIDEO_FPS          = 30     # 輸出 MP4 fps
os.makedirs(VIZ_VIDEO_DIR, exist_ok=True)

# === 依賴：lerobot 內建 helper ===
from lerobot.rewards.sarm.compute_rabc_weights import (
    load_sarm_resources,
    to_numpy_image,
    interpolate_progress,
)

# === 載入 dataset + model（已 cache，只 re-init） ===
with progress_stage(
    f'載入 SARM 資源（dataset={ANNOTATED_DATASET}, model={MODEL_REPO_ID}）',
    steps=['SARMRewardModel.from_pretrained',
           'LeRobotDataset（帶 delta_timestamps）',
           'make_sarm_pre_post_processors'],
    heartbeat_seconds=30,
):
    dataset, reward_model, preprocess = load_sarm_resources(
        ANNOTATED_DATASET, MODEL_REPO_ID, device='cuda',
    )

image_key = reward_model.config.image_key
state_key = reward_model.config.state_key
target_idx = reward_model.config.n_obs_steps // 2
dual_mode  = reward_model.config.uses_dual_heads
head_mode  = HEAD_MODE if (HEAD_MODE in ('sparse','dense') and (HEAD_MODE=='sparse' or dual_mode)) else 'sparse'
num_stages = getattr(reward_model.config, f'num_{head_mode}_stages')
stage_labels_cfg = getattr(reward_model.config, f'{head_mode}_subtask_names') \
                    or [f'Stage {i+1}' for i in range(num_stages)]
print(f'  head_mode={head_mode}, num_stages={num_stages}')

# preprocess eval mode（停掉 augmentation）
if hasattr(preprocess, 'eval'):
    preprocess.eval()
for step in getattr(preprocess, 'steps', []):
    if hasattr(step, 'eval'):
        step.eval()


def render_episode_mp4(frames, progress_preds, stage_preds, title, mp4_path,
                       stage_labels, fps=30):
    """渲染左影片右進度條的 MP4。frames 為 list/ndarray of (H, W, 3) uint8。"""
    import imageio
    from PIL import Image, ImageDraw, ImageFont
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    n = len(frames)
    if n == 0:
        return
    fh, fw = frames[0].shape[:2]
    panel_w = max(fw, 480)
    canvas_h = fh
    canvas_w = fw + panel_w

    # --- 一次性 render 整段預測曲線當背景 ---
    dpi = 100
    fig_h_in = canvas_h / dpi
    fig_w_in = panel_w / dpi
    fig, ax_p = plt.subplots(figsize=(fig_w_in, fig_h_in * 0.55), dpi=dpi)
    fig.patch.set_facecolor('white')
    x = np.arange(n)
    ax_p.plot(x, progress_preds, color='#2E86AB', linewidth=2)
    ax_p.fill_between(x, 0, progress_preds, alpha=0.3, color='#2E86AB')
    ax_p.set_ylim(-0.05, 1.1)
    ax_p.set_xlim(0, max(n - 1, 1))
    ax_p.set_ylabel('Progress')
    ax_p.set_xlabel('Frame')
    ax_p.grid(True, alpha=0.3)
    ax_p.set_title('Predicted progress', fontsize=10)
    fig.tight_layout()
    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba() if hasattr(fig.canvas, 'buffer_rgba') else fig.canvas.tostring_rgb()
    if hasattr(fig.canvas, 'buffer_rgba'):
        curve_rgba = np.asarray(buf)
        curve_rgb = curve_rgba[..., :3].copy()
    else:
        w_px, h_px = fig.canvas.get_width_height()
        curve_rgb = np.frombuffer(buf, dtype=np.uint8).reshape(h_px, w_px, 3)
    # axes pixel bbox（轉成 canvas-relative）
    bbox = ax_p.get_window_extent()
    ax_x0_in_curve = int(bbox.x0)
    ax_x1_in_curve = int(bbox.x1)
    ax_y0_in_curve = int(curve_rgb.shape[0] - bbox.y1)
    ax_y1_in_curve = int(curve_rgb.shape[0] - bbox.y0)
    plt.close(fig)

    # resize curve panel 到 (panel_w, target_h)
    target_curve_h = int(canvas_h * 0.55)
    curve_img = Image.fromarray(curve_rgb).resize((panel_w, target_curve_h))
    scale_x = panel_w / curve_rgb.shape[1]
    scale_y = target_curve_h / curve_rgb.shape[0]
    ax_x0 = int(ax_x0_in_curve * scale_x)
    ax_x1 = int(ax_x1_in_curve * scale_x)
    ax_y0 = int(ax_y0_in_curve * scale_y)
    ax_y1 = int(ax_y1_in_curve * scale_y)

    # 字體
    try:
        font     = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 16)
        font_sm  = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 12)
        font_tt  = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 13)
    except Exception:
        font = font_sm = font_tt = ImageFont.load_default()

    # 進度條/曲線/字幕 layout（在 panel 內的座標）
    bar_top    = 12
    bar_h      = 22
    bar_left   = 14
    bar_right  = panel_w - 14
    title_y    = bar_top + bar_h + 6
    curve_y0   = title_y + 30
    curve_y1   = curve_y0 + target_curve_h
    subtask_y  = canvas_h - 40

    writer = imageio.get_writer(mp4_path, fps=fps, codec='libx264', quality=8,
                                macro_block_size=1)
    try:
        for i in range(n):
            canvas = Image.new('RGB', (canvas_w, canvas_h), 'white')
            # left: video frame
            vf = frames[i]
            if vf.ndim == 2:
                vf = np.stack([vf]*3, axis=-1)
            elif vf.shape[-1] == 1:
                vf = np.repeat(vf, 3, axis=-1)
            canvas.paste(Image.fromarray(vf.astype(np.uint8)), (0, 0))
            # right: curve background（貼在 curve_y0）
            canvas.paste(curve_img, (fw, curve_y0))

            draw = ImageDraw.Draw(canvas)
            p = float(np.clip(progress_preds[i], 0, 1))

            # 水平 progress bar
            draw.rectangle([(fw + bar_left, bar_top),
                            (fw + bar_right, bar_top + bar_h)],
                           outline=(80, 80, 80), fill=(235, 235, 235), width=1)
            fill_x = fw + bar_left + int((bar_right - bar_left) * p)
            draw.rectangle([(fw + bar_left, bar_top),
                            (fill_x, bar_top + bar_h)],
                           outline=None, fill=(46, 134, 171))
            draw.text((fw + bar_left, title_y),
                      f'Progress: {p:.3f}', fill=(0, 0, 0), font=font_tt)

            # 在 curve 上畫垂直 marker line
            marker_x = fw + ax_x0 + int(i / max(n - 1, 1) * (ax_x1 - ax_x0))
            draw.line([(marker_x, curve_y0 + ax_y0),
                       (marker_x, curve_y0 + ax_y1)],
                      fill=(0, 0, 0), width=2)
            # 在 marker 上畫小圓點代表當前 progress
            dot_y = curve_y0 + ax_y1 - int((ax_y1 - ax_y0) * (p / 1.1 + 0.05/1.1))
            r = 4
            draw.ellipse([(marker_x - r, dot_y - r), (marker_x + r, dot_y + r)],
                         fill=(220, 40, 40), outline=(0, 0, 0))

            # 當前 subtask
            stage_idx = int(np.argmax(stage_preds[i]))
            label = stage_labels[stage_idx]
            if len(label) > 50:
                label = label[:50] + '...'
            draw.text((fw + bar_left, subtask_y),
                      f'Subtask [{stage_idx}]: {label}',
                      fill=(0, 0, 0), font=font_sm)

            # frame counter（疊在左下角，黑底白字方便看）
            counter = f'Frame {i+1}/{n}'
            tw_box = (10, canvas_h - 28, 10 + 130, canvas_h - 6)
            draw.rectangle(tw_box, fill=(0, 0, 0))
            draw.text((tw_box[0] + 6, tw_box[1] + 4), counter,
                      fill=(255, 255, 255), font=font_sm)

            # episode title 疊在左上
            title_short = title[:48] + ('...' if len(title) > 48 else '')
            draw.rectangle((6, 6, 6 + min(360, fw - 12), 28), fill=(0, 0, 0))
            draw.text((10, 8), title_short, fill=(255, 255, 255), font=font_sm)

            writer.append_data(np.array(canvas))
    finally:
        writer.close()


def run_inference_for_episode(episode_idx, stride):
    """Returns (full_frames, progress_per_frame, stage_per_frame)."""
    ep = dataset.meta.episodes[episode_idx]
    ep_start = int(ep['dataset_from_index'])
    ep_end   = int(ep['dataset_to_index'])
    task     = dataset[ep_start].get('task', 'perform the task')
    n_frames = ep_end - ep_start

    # 推論 anchor frame 列表（每 stride 個 + 最後一個）
    infer_local = list(range(0, n_frames, stride))
    if (n_frames - 1) not in infer_local:
        infer_local.append(n_frames - 1)
    infer_local = sorted(set(infer_local))

    prog = np.full(n_frames, np.nan, dtype=np.float32)
    stages = np.full((n_frames, num_stages), np.nan, dtype=np.float32)
    full_frames = [None] * n_frames

    for local_i in tqdm(range(n_frames), desc=f'Episode {episode_idx} frames', leave=False):
        f_idx = ep_start + local_i
        sample = dataset[f_idx]
        full_frames[local_i] = to_numpy_image(sample[image_key])
        if local_i not in infer_local:
            continue

        batch = {
            image_key: sample[image_key],
            'task': task,
            'index': f_idx,
            'episode_index': episode_idx,
        }
        if state_key in sample:
            batch[state_key] = sample[state_key]

        with torch.no_grad():
            processed = preprocess(batch)
            video_features = processed['video_features'].to(reward_model.device)
            text_features  = processed['text_features'].to(reward_model.device)
            state_features = processed.get('state_features')
            if state_features is not None:
                state_features = state_features.to(reward_model.device)
            lengths = processed.get('lengths')

            reward, stage_probs = reward_model.calculate_rewards(
                text_embeddings=text_features,
                video_embeddings=video_features,
                state_features=state_features,
                lengths=lengths,
                return_all_frames=True,
                return_stages=True,
                head_mode=head_mode,
            )
            if isinstance(reward, torch.Tensor):
                reward = reward.cpu().numpy()
                stage_probs = stage_probs.cpu().numpy()

            if reward.ndim == 2:
                prog[local_i] = reward[0, target_idx]
                stages[local_i] = stage_probs[0, target_idx, :]
            else:
                prog[local_i] = reward[target_idx]
                stages[local_i] = stage_probs[target_idx, :]

        torch.cuda.empty_cache()

    # 內插
    valid = np.isfinite(prog)
    valid_idx = np.where(valid)[0]
    all_idx = np.arange(n_frames)
    if valid_idx.size >= 1:
        prog_interp = interpolate_progress(valid_idx, prog[valid_idx], all_idx)
        stage_interp = np.zeros_like(stages)
        for s in range(num_stages):
            stage_interp[:, s] = interpolate_progress(valid_idx, stages[valid_idx, s], all_idx)
        # renormalize
        row_sums = stage_interp.sum(axis=1, keepdims=True)
        nz = row_sums.squeeze(-1) > 0
        stage_interp[nz] = stage_interp[nz] / row_sums[nz]
    else:
        prog_interp = np.nan_to_num(prog, nan=0.0)
        stage_interp = np.nan_to_num(stages, nan=0.0)

    return np.stack(full_frames, axis=0), prog_interp, stage_interp, task


# === 主流程：對 EVAL_EPISODES 內每個 episode 各自存 MP4 ===
viz_episodes = [e for e in EVAL_EPISODES if 0 <= e < dataset.num_episodes]
skipped = [e for e in EVAL_EPISODES if e not in viz_episodes]
if skipped:
    print(f'⚠ 跳過超出範圍的 episode（dataset 共 {dataset.num_episodes} 個）: {skipped}')
if not viz_episodes:
    raise RuntimeError(f'EVAL_EPISODES 中沒有任何有效 episode（dataset 只有 {dataset.num_episodes} 個）')

with progress_stage(
    f'渲染 {len(viz_episodes)} 個 episode 的「影片+進度條」MP4（stride={VIDEO_STRIDE}）',
    steps=[f'episodes={viz_episodes}',
           f'每個 episode 推論 + 抽幀 + 渲染 MP4',
           f'輸出到 {VIZ_VIDEO_DIR}'],
    heartbeat_seconds=30,
):
    for ep_idx in viz_episodes:
        full_frames, prog, stages, task = run_inference_for_episode(ep_idx, VIDEO_STRIDE)
        mp4_path = os.path.join(VIZ_VIDEO_DIR, f'sarm_video_ep{ep_idx}_{head_mode}.mp4')
        render_episode_mp4(
            frames=full_frames,
            progress_preds=prog,
            stage_preds=stages,
            title=f'{task} (ep {ep_idx})',
            mp4_path=mp4_path,
            stage_labels=stage_labels_cfg,
            fps=VIDEO_FPS,
        )
        print(f'  ✓ ep {ep_idx}: {mp4_path}')

print(f'\n所有 MP4 存放於：{VIZ_VIDEO_DIR}')


In [ ]:
# Cell 14d: 在 notebook 內嵌入剛剛產生的 MP4
import glob, os
from IPython.display import Video, display

video_files = sorted(glob.glob(f'{VIZ_VIDEO_DIR}/*.mp4'))
if not video_files:
    print('找不到 MP4，請確認 Cell 14c 成功執行')
else:
    print(f'找到 {len(video_files)} 支 MP4：')
    for p in video_files:
        print(f'\n{os.path.basename(p)} ({os.path.getsize(p)/1e6:.1f} MB):')
        display(Video(p, embed=True, width=900))


In [ ]:
# Cell 14e: 把 /content/sarm_predictions_video 整個打包下載到本機
# ============================================================
# 在 Colab 跑這個 cell 會：
#   1) 把 VIZ_VIDEO_DIR 打成 zip
#   2) 觸發瀏覽器下載對話框，存到你電腦的「下載」資料夾
# 如果檔案大（>1GB）瀏覽器下載比較容易斷，建議改用 Google Drive 那段。
# ============================================================
import os, shutil, glob

zip_base = '/content/sarm_predictions_video'   # zip 名稱（不含 .zip）
src_dir  = VIZ_VIDEO_DIR

mp4_files = sorted(glob.glob(f'{src_dir}/*.mp4'))
if not mp4_files:
    raise RuntimeError(f'{src_dir} 內沒有 MP4，請先跑 Cell 14c')

print(f'準備打包 {len(mp4_files)} 支影片：')
total_mb = 0
for p in mp4_files:
    size_mb = os.path.getsize(p) / 1e6
    total_mb += size_mb
    print(f'  - {os.path.basename(p)}  ({size_mb:.1f} MB)')
print(f'總大小: {total_mb:.1f} MB')

zip_path = shutil.make_archive(zip_base, 'zip', root_dir=src_dir)
print(f'\n✓ 已打包: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')

# 觸發 Colab 瀏覽器下載
try:
    from google.colab import files
    files.download(zip_path)
    print('✓ 已觸發瀏覽器下載（檢查瀏覽器下載列）')
except ImportError:
    print('（不是 Colab 環境，跳過 files.download；zip 在上面那個路徑）')

# === 備案：存到 Google Drive（適合大檔 / 多次下載）===
# 把下面三行取消註解再跑：
#
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(zip_path, '/content/drive/MyDrive/sarm_predictions_video.zip')
# print('✓ 已複製到 /content/drive/MyDrive/sarm_predictions_video.zip')
